In [1]:
import pandas as pd
import gc, re, csv, random
from collections import defaultdict
from pathlib import Path

BASE = "/content/drive/MyDrive/pandora"
COMMENTS = "/content/drive/MyDrive/PANDORA/pandora_comments/all_comments_since_2015.csv"
PROFILES = "/content/drive/MyDrive/PANDORA/pandora_profiles/author_profiles.csv"

WORK = f"{BASE}/work_transformer_opt"
Path(WORK).mkdir(exist_ok=True)

PASS2_OUT = f"{WORK}/eligible_comments.csv"
FINAL_OUT = f"{BASE}/pandora_user_level_transformer_opt.parquet"

CHUNK = 200_000
MIN_WORDS = 500        # stronger linguistic threshold
MAX_CHARS = 150_000    # allow more context

OCEAN = ["openness","conscientiousness","extraversion","agreeableness","neuroticism"]

In [2]:
profiles = pd.read_csv(PROFILES)
profiles = profiles.dropna(subset=OCEAN)
valid_authors = set(profiles.author)

print("Profiles with full OCEAN:", len(valid_authors))

Profiles with full OCEAN: 1568


In [3]:
URL_RE = re.compile(r"http\S+|www\S+")
SPACE = re.compile(r"\s+")
CONTROL = re.compile(r"[\x00-\x1f\x7f]")  # remove control chars only

def transformer_clean(text):
    text = str(text)

    # remove URLs only
    text = URL_RE.sub(" ", text)

    # remove control characters
    text = CONTROL.sub(" ", text)

    # normalize whitespace
    text = SPACE.sub(" ", text)

    return text.strip()

In [4]:
author_wc = defaultdict(int)

reader = pd.read_csv(
    COMMENTS,
    usecols=["author","body","lang"],
    chunksize=CHUNK
)

for i,c in enumerate(reader,1):
    c = c[(c.lang=="en") & (c.author.isin(valid_authors))]

    for a,t in zip(c.author,c.body):
        author_wc[a] += len(str(t).split())

    del c
    gc.collect()

eligible_authors = {a for a,w in author_wc.items() if w >= MIN_WORDS}
print("Eligible authors:", len(eligible_authors))

Eligible authors: 1416


In [5]:
author_text = defaultdict(list)

reader = pd.read_csv(
    COMMENTS,
    usecols=["author","body","lang"],
    chunksize=CHUNK
)

for c in reader:
    c = c[(c.lang=="en") & (c.author.isin(eligible_authors))]

    for a,t in zip(c.author,c.body):
        txt = transformer_clean(t)
        if len(txt.split()) >= 5:
            author_text[a].append(txt)

    del c
    gc.collect()

In [6]:
rows = []

for a, parts in author_text.items():
    random.shuffle(parts)  # important

    full_text = " ".join(parts)

    if len(full_text) > MAX_CHARS:
        full_text = full_text[:MAX_CHARS]

    rows.append({
        "author": a,
        "text": full_text
    })

text_df = pd.DataFrame(rows)

In [7]:
final = text_df.merge(profiles, on="author", how="inner")
final.to_parquet(FINAL_OUT, index=False)

print("Saved:", FINAL_OUT)
print("Users:", len(final))

Saved: /content/drive/MyDrive/pandora/pandora_user_level_transformer_opt.parquet
Users: 1416


In [8]:
df = final.copy()
df["word_count"] = df.text.str.split().str.len()

print("Avg words/user:", int(df.word_count.mean()))
print("Min words:", df.word_count.min())
print("Max words:", df.word_count.max())

print(df[OCEAN].describe())

Avg words/user: 16329
Min words: 496
Max words: 29849
          openness  conscientiousness  extraversion  agreeableness  \
count  1416.000000        1416.000000   1416.000000    1416.000000   
mean     62.371116          40.011299     37.242232      42.112288   
std      27.648516          30.296063     30.338205      30.912194   
min       0.000000           0.000000      0.000000       0.000000   
25%      43.000000          13.000000     10.000000      13.000000   
50%      69.000000          35.000000     30.000000      39.000000   
75%      85.000000          65.000000     60.000000      70.000000   
max     100.000000          99.000000     99.000000     100.000000   

       neuroticism  
count  1416.000000  
mean     49.458333  
std      32.245300  
min       0.000000  
25%      17.750000  
50%      49.000000  
75%      81.000000  
max     100.000000  
